# F6-svd-spectral — Session 3: The Singular Value Decomposition

*One class session, roughly 85 minutes. Prerequisites: Sessions 1–2 of
this unit (eigenpairs, spectral decomposition, the reorder and sign
pins) and F3 (Gram matrices, rank, outer products).*

**This session:** the decomposition that works for **every** matrix,
square or not — $W = U \Sigma V^{\mathsf T}$; the call
`np.linalg.svd` and this course's tool-legality convention; exact
shapes for tall and wide matrices in both the **thin** and **full**
variants (the shape table); and the unit's load-bearing theorem — the
**bridge**: the SVD of $W$ hands you the complete spectral
decomposition of the Gram matrix $S = WW^{\mathsf T}$, in a thin form
(nonzero spectrum only) and a full form (zero-padded, whole-space).
Plus a fully worked constrained-coding example.

This closes the first sitting of this double unit (Sessions 1–3);
Sessions 4–5 are designed as a second sitting.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np

SEED = 20260804

## 1. Square Tools Hit a Rectangular Wall

Sessions 1–2 built a beautiful machine for **square symmetric**
matrices.
But the data tables you actually meet are rectangular: $200$
measurement records over $40$ channels, $6$ samples in $4$ variables —
shape $(n, d)$ with $n \ne d$.
For a rectangular $W$ the eigen-question does not even parse:
$Wq = \lambda q$ needs input and output to live in the same space, and
a $(6, 4)$ matrix eats 4-vectors but emits 6-vectors.

Yet the *questions* eigenanalysis answered still make sense for
rectangular matrices:

- Which input directions does $W$ stretch the most?
- What are the stretch factors?
- Can $W$ be rebuilt from a few rank-1 atoms (F3), and which atoms
  matter most?

The **singular value decomposition (SVD)** answers all three, for every
matrix, with no symmetry or squareness asked.
The price of full generality is modest: *two* orthonormal vector
families instead of one — one family in the input space, one in the
output space.

In [ ]:
rng = np.random.default_rng(SEED)
W = rng.normal(0, 1, (6, 4))       # this session's running example
print("W shape:", W.shape, "-> maps 4-vectors to 6-vectors")

### Checkpoint 1

1. Why is $Wq = \lambda q$ meaningless for a $(6, 4)$ matrix?
   One sentence about shapes.
2. Both $WW^{\mathsf T}$ and $W^{\mathsf T}W$ *are* square and
   symmetric.
   What are their shapes for $W\ (6, 4)$, and what does F3 call the
   first one?

## 2. Reading $W = U \Sigma V^{\mathsf T}$

> **Fact (stated; existence proof out of scope — using it is this
> unit's business).**
> Every matrix $W\ (n, d)$ factors as
> $$W = U\,\Sigma\,V^{\mathsf T}$$
> where $U$ and $V$ have **orthonormal columns** ($u_i$ in the output
> space $\mathbb{R}^n$, $v_i$ in the input space $\mathbb{R}^d$), and
> $\Sigma$ is diagonal with non-negative entries in **descending**
> order:
> $\sigma_1 \ge \sigma_2 \ge \cdots \ge \sigma_r \ge \cdots \ge 0$ —
> the **singular values**.

Equivalently, as an outer-product atom sum (F3's favorite form):

$$W = \sum_k \sigma_k \, u_k v_k^{\mathsf T},$$

and as an action recipe, for each pair:

$$W v_k = \sigma_k u_k
\qquad\text{and}\qquad
W^{\mathsf T} u_k = \sigma_k v_k.$$

**The machine reading** (right to left, like Session 2 §3):
$V^{\mathsf T}x$ re-expresses the input in the $v$-basis (rotation);
$\Sigma$ stretches coordinate $k$ by $\sigma_k \ge 0$;
$U$ carries the result into the output space along the $u$-basis
(rotation).
*Every matrix is rotate → stretch → rotate.*
The perpendicular input directions $v_k$ are stretched by $\sigma_k$
and land on the perpendicular output directions $u_k$; nothing else
about the map exists.

**Conventions built into the definition** — each one will matter:

1. **$\sigma_k \ge 0$ always.** Negative stretch does not occur;
   a sign is absorbed by flipping $u_k$ or $v_k$ (a direction choice,
   not a magnitude).
   Hand-check on $W = \begin{pmatrix} 3 & 0 \\ 0 & -2 \end{pmatrix}$:
   $\sigma = (3, 2)$, with $W = U\Sigma V^{\mathsf T}$, $U = I$,
   $\Sigma = \mathrm{diag}(3, 2)$,
   $V = \mathrm{diag}(1, -1)$ — the minus sign moved into $V$.
2. **Descending order** — matching this course's eigenvalue pin, and
   (unlike `eig`/`eigh`!) matching what `np.linalg.svd` itself
   returns.
3. **$\operatorname{rank}(W) = \#\{k : \sigma_k > 0\}$** — the atom
   sum has exactly that many genuinely contributing rank-1 atoms
   (F3's minimal-decomposition count, now with a canonical choice of
   atoms).

### Checkpoint 2

1. For $W = \begin{pmatrix} 1 & 2 \\ 2 & 4 \end{pmatrix}$
   (rows proportional — rank 1 by F3), what must its singular values
   be, given that $\sum_k \sigma_k^2$ turns out to equal the sum of
   squared entries?
   (Peek ahead logic: Session 4 proves that identity; here, count
   nonzeros via the rank rule, then solve.)
2. Using $Wv_k = \sigma_k u_k$: what is $\lVert W v_k \rVert$, and why
   does this justify calling $\sigma_1$ "the largest stretch the map
   applies to any unit input direction"?
3. In the hand-check of convention 1, verify
   $U \Sigma V^{\mathsf T} = W$ by explicit multiplication.

## 3. `np.linalg.svd` — and How Tool Legality Works From Here On

```python
U, s, Vt = np.linalg.svd(W, full_matrices=False)
```

Read the outputs carefully — two of the three are shaped by
convention:

- `U (n, k)` — left singular vectors as **columns**.
- `s (k,)` — the singular values as a **vector** (not a diagonal
  matrix!), **already descending** — no reorder idiom needed here.
- `Vt (k, d)` — **$V^{\mathsf T}$, already transposed**: the right
  singular vectors are the **ROWS** of `Vt`.
  `Vt[k]` is $v_k^{\mathsf T}$; if you want $V$ itself, that is
  `Vt.T`.

(with $k = \min(n, d)$ in this thin form — shapes get their own
section next.)
To rebuild: `W ≈ U @ np.diag(s) @ Vt`.

**The legality convention, stated once for the whole unit.**
Earlier units banned `np.linalg` calls inside specific problems — F3's
banned-`@` register and its cousins.
Those bans were **per-problem skill-forcing devices, not a
curriculum-wide rule**, and this unit is where the flip happens: the
exam itself scopes tool legality per problem — one part bans
`np.linalg` outright to force a hand-built normalization, while
another part *hints* that `np.linalg.svd` is the intended tool.
Mirroring that register: **from F6 onward, `np.linalg.svd`,
`np.linalg.eig`, `np.linalg.eigh`, and `np.linalg.norm` are legal
wherever the problem's statement allows them, and every F6 practice
statement says explicitly which `np.linalg` calls are allowed** —
read that line first, every time, exactly as you would on the paper.

In [ ]:
U, s, Vt = np.linalg.svd(W, full_matrices=False)
print("shapes: U", U.shape, " s", s.shape, " Vt", Vt.shape)
print("s (descending, non-negative):", s)

recon_gap = np.abs(U @ np.diag(s) @ Vt - W).max()
print("reconstruction gap:", recon_gap)

# Orthonormal columns, both families (F2 test):
print("max |U^T U - I|:", np.abs(U.T @ U - np.eye(4)).max())
print("max |V^T V - I|:", np.abs(Vt @ Vt.T - np.eye(4)).max())

# The action recipe, pair by pair: W v_k = sigma_k u_k
action_gap = np.abs(W @ Vt.T - U * s).max()
print("max |W v_k - sigma_k u_k|:", action_gap)

(The last line reuses Session 2 §7's broadcasting idiom: column $k$ of
`W @ Vt.T` is $Wv_k$, and `U * s` scales column $k$ of $U$ by
$\sigma_k$.)

### Checkpoint 3

1. Why is `Vt @ Vt.T` — not `Vt.T @ Vt` — the correct orthonormality
   check for the *right singular vectors* in the thin form here?
   (Which one is $V^{\mathsf T}V$ in math notation?)
2. A script builds the reconstruction as `U @ np.diag(s) @ Vt.T`.
   What did it actually compute, and how does the reconstruction gap
   react?
3. State, from memory, which of the three outputs of `np.linalg.svd`
   needs a reorder idiom to meet the course's descending convention.

## 4. Shapes: Tall vs Wide, Thin vs Full

`np.linalg.svd` has a switch, and NumPy's *default* is the opposite of
this course's default call:

- **`full_matrices=False` — the THIN SVD (our default call).**
  Only the $k = \min(n, d)$ vector pairs that can carry nonzero
  singular values.
  Cheap, and everything reconstruction-related needs nothing more.
- **`full_matrices=True` — the FULL SVD (NumPy's default!).**
  $U$ is completed to a square $(n, n)$ orthonormal basis of the
  *whole* output space, $V^{\mathsf T}$ to a square $(d, d)$ basis of
  the input space; `s` still has only $k$ entries.
  The extra columns of $U$ are perpendicular to everything $W$ can
  produce — they matter exactly when you need a whole-space statement
  (the bridge's full form, two sections from now).

**The shape table** (memorize the tall column; reconstruct the wide
one by the symmetry $n \leftrightarrow d$):

| $W\ (n, d)$ | call | `U` | `s` | `Vt` |
|---|---|---|---|---|
| tall, $n > d$ | thin (`full_matrices=False`) | $(n, d)$ | $(d,)$ | $(d, d)$ |
| tall, $n > d$ | full (`full_matrices=True`) | $(n, n)$ | $(d,)$ | $(d, d)$ |
| wide, $n < d$ | thin | $(n, n)$ | $(n,)$ | $(n, d)$ |
| wide, $n < d$ | full | $(n, n)$ | $(n,)$ | $(d, d)$ |

Note what stays small: `s` always has $\min(n, d)$ entries — a matrix
cannot have more stretch factors than its smaller dimension (F3's rank
bound wearing SVD clothes).

In [ ]:
# Tall example: our running (6, 4) matrix
for fm in [False, True]:
    U_, s_, Vt_ = np.linalg.svd(W, full_matrices=fm)
    print(f"tall (6,4), full_matrices={fm}:  U {U_.shape}  s {s_.shape}  Vt {Vt_.shape}")

# Wide example: a hand-sized (3, 5) matrix
W_wide = np.array([[1., 0., 2., 0., 2.],
                   [0., 3., 0., 1., 0.],
                   [2., 0., 1., 0., 1.]])
for fm in [False, True]:
    U_, s_, Vt_ = np.linalg.svd(W_wide, full_matrices=fm)
    print(f"wide (3,5), full_matrices={fm}:  U {U_.shape}  s {s_.shape}  Vt {Vt_.shape}")

**Thin $U$ is NOT a whole-space basis.**
For the tall thin case, $U^{\mathsf T}U = I_d$ (its 4 columns are
orthonormal) — but $UU^{\mathsf T} \ne I_n$: four orthonormal columns
cannot span $\mathbb{R}^6$.
This asymmetry is about to become the whole point of the bridge's two
forms:

In [ ]:
U, s, Vt = np.linalg.svd(W, full_matrices=False)
Uf, sf, Vtf = np.linalg.svd(W, full_matrices=True)

print("thin: max |U^T U - I_4|:", np.abs(U.T @ U - np.eye(4)).max())
print("thin: max |U U^T - I_6|:", np.abs(U @ U.T - np.eye(6)).max(), " <- NOT identity")
print("full: max |U U^T - I_6|:", np.abs(Uf @ Uf.T - np.eye(6)).max(), " <- square orthonormal")

### Checkpoint 4

1. Without running code: shapes of `U`, `s`, `Vt` for a $(100, 7)$
   matrix under each of `full_matrices=False` and `True`?
2. Same question for $(7, 100)$.
3. Why can a $(6, 4)$ matrix have at most $4$ nonzero singular values?
   Answer once with the shape table and once with F3's rank bound.

## 5. The Bridge, Derived: the SVD of $W$ Solves the Spectral Problem
for $S = WW^{\mathsf T}$

Here is the theorem this unit is named for.
It says the two halves of the unit — eigen-world (Sessions 1–2) and
SVD-world (this session) — are one machine.

**Setup.** $W\ (n, d)$ with $n > d$, thin SVD
$W = \sum_k \sigma_k u_k v_k^{\mathsf T}$, and the Gram matrix
$S = WW^{\mathsf T}\ (n, n)$ — symmetric and PSD (Session 2 §6).

**Derivation, in component form — two orthonormality strikes.**
First, hit $u_i$ with $W^{\mathsf T}$ (the action recipe, derived
rather than quoted):

$$
W^{\mathsf T} u_i
= \Bigl(\sum_k \sigma_k u_k v_k^{\mathsf T}\Bigr)^{\mathsf T} u_i
= \sum_k \sigma_k v_k \underbrace{(u_k \cdot u_i)}_{=\,0 \text{ unless } k=i}
= \sigma_i v_i
$$

— the sum collapses because the $u_k$ are orthonormal (F2 dot
products: $u_k \cdot u_i = 1$ if $k = i$, else $0$).
Now hit the result with $W$:

$$
S u_i = W\,(W^{\mathsf T} u_i) = W\,(\sigma_i v_i)
= \sigma_i \sum_k \sigma_k u_k \underbrace{(v_k \cdot v_i)}_{=\,0 \text{ unless } k=i}
= \sigma_i^2\, u_i.
$$

**Conclusion:** each left singular vector $u_i$ is an eigenvector of
$S$, with eigenvalue $\lambda_i = \sigma_i^2$. ∎

That single line explains a Session 2 mystery *and* inherits all the
right properties: eigenvalues $\sigma_i^2 \ge 0$ — Gram matrices are
PSD because their eigenvalues are literally squares.

**But read the fine print — the bridge has two forms.**
$S$ is $(n, n)$: it owns $n$ eigenvalues.
The derivation above only produced $d$ of them.

> **THIN form** (`full_matrices=False`, $U\ (n, d)$).
> The columns of thin $U$ are eigenvectors for the (at most) $d$
> **nonzero** eigenvalues $\lambda_i = \sigma_i^2$.
> The remaining $n - d$ eigenvalues of $S$ are all $0$, and their
> eigenvectors are **NOT in thin $U$** — thin $U$ cannot span $S$'s
> null space (Section 4: $UU^{\mathsf T} \ne I_n$).
> Thin form = the nonzero spectrum only.

> **FULL form** (`full_matrices=True`, $U\ (n, n)$).
> Take $Q = U$ (full) and zero-pad the eigenvalues to length $n$:
> $$\lambda = (\sigma_1^2, \dots, \sigma_d^2,
> \underbrace{0, \dots, 0}_{n-d}).$$
> Then $S = Q\,\Lambda\,Q^{\mathsf T}$ is the **complete spectral
> decomposition**: the extra $n - d$ columns of full $U$ are
> orthonormal, perpendicular to every output $W$ can make, and satisfy
> $S u = W(W^{\mathsf T}u) = W\,0 = 0$ — genuine $\lambda = 0$
> eigenvectors.

**The mirror bridge.** The same two strikes applied on the other side
give $(W^{\mathsf T}W)\,v_i = \sigma_i^2\,v_i$: the right singular
vectors are the eigenvectors of the $(d, d)$ Gram-of-columns matrix,
with the *same* nonzero eigenvalues $\sigma_i^2$.

### Checkpoint 5

1. In the first strike, say precisely where orthonormality of the
   $u_k$ was used, and what the sum would look like without it.
2. $W$ is $(200, 40)$ with all $\sigma_i > 0$.
   How many eigenvalues does $S = WW^{\mathsf T}$ have, what are they,
   and what multiplicity does $0$ carry?
3. Why is *thin* $U$ structurally incapable of providing the
   $\lambda = 0$ eigenvectors?
   (What space would its columns have to span?)

## 6. The Bridge, Numerically: Both Forms Verified

Route A: `eigh` on $S$ directly (Session 2 machinery, reorder idiom).
Route B: SVD of $W$, then the bridge.
They must agree — thin form on the nonzero spectrum, full form on the
whole decomposition.

In [ ]:
S = W @ W.T                          # (6, 6) Gram matrix, rank <= 4

# Route A: spectral, direct
lam_asc, Q_asc = np.linalg.eigh(S)
lam, Q = lam_asc[::-1], Q_asc[:, ::-1]      # the pinned reorder
print("eigh eigenvalues (desc):", lam)

# Route B: bridge from the SVD
U, s, Vt = np.linalg.svd(W, full_matrices=False)
print("sigma^2               :", s**2)

# THIN form: nonzero spectrum only
bridge_gap = np.abs(lam[:4] - s**2).max()
tail_max = np.abs(lam[4:]).max()
print("thin-form gap (top 4):", bridge_gap)
print("tail |eigenvalues| (should be ~0):", tail_max)

In [ ]:
# FULL form: zero-pad the spectrum, use the FULL U
Uf = np.linalg.svd(W, full_matrices=True)[0]
lam_pad = np.concatenate([s**2, np.zeros(6 - 4)])
S_rebuilt = Uf @ np.diag(lam_pad) @ Uf.T
print("full-form reconstruction gap:", np.abs(S_rebuilt - S).max())

# and the mirror bridge on W^T W:
lam_v = np.linalg.eigh(W.T @ W)[0][::-1]
print("eigh(W^T W) desc:", lam_v)
print("mirror bridge gap:", np.abs(lam_v - s**2).max())

Every gap at machine precision: the SVD of the rectangular $W$ really
does carry the complete eigen-story of both square Gram matrices.

One caution, foreshadowing the capstone: we verified the full form by
its **reconstruction** — an invariant — and *not* by comparing full
$U$'s tail columns entrywise against `eigh`'s tail columns.
Those tail eigenvectors belong to the $(n-d)$-fold **repeated
eigenvalue $0$**, where Session 2's sign pin explicitly refuses
individual-vector comparisons: within a degenerate block, each route
may return *any* rotation of any valid basis, and no sign-fix can
reconcile two arbitrary rotations.
Reconstruction gaps, eigen-equation residuals, and projector
comparisons are the honest checks there — Session 5 makes this the
capstone's verification contract.

### Checkpoint 6

1. Predict before running: for the wide matrix `W_wide (3, 5)` of
   Section 4, what does the thin/full distinction look like on the
   *other* side — which Gram matrix ($W W^{\mathsf T}$ or
   $W^{\mathsf T} W$) needs zero-padding, and how many padded zeros?
2. Reviewing the printed spectra: why does `eigh(S)` show two
   eigenvalues at $\sim 10^{-16}$ rather than exactly $0$, and which
   Session 2 footnote covers this?

## 7. Worked Exam-Style Example: Bridge Verification Under Constraints

---

**Worked exam-style example 3 (constrained coding).**

> Write `bridge_check(W)` for a tall matrix `W (n, d)`, `n > d`,
> returning `(bridge_gap, tail_max, pad_gap)` where
> `bridge_gap` = max abs difference between the top-$d$ descending
> eigenvalues of $S = WW^{\mathsf T}$ and the squared singular values
> of `W`;
> `tail_max` = max absolute value of the remaining $n - d$
> eigenvalues;
> `pad_gap` = max absolute entry of
> $U_{\text{full}}\,\mathrm{diag}(\lambda_{\text{pad}})\,
> U_{\text{full}}^{\mathsf T} - S$ for the zero-padded spectrum.
> **Allowed `np.linalg` calls: `np.linalg.svd` and `np.linalg.eigh`
> only.**
> All three outputs must be $< 10^{-8}$ except `tail_max`, which must
> be $< 10^{-10}$, on the seeded test matrix.
> Reasoning is not required.

*Solution, spelled out.*
Plan: `eigh` + pinned reorder for route A; one `svd` call *with*
`full_matrices=True` — note the full `U` is needed for `pad_gap`, but
`s` is the same either way; assemble the three gaps exactly as
defined.

In [ ]:
def bridge_check(W):
    n, d = W.shape
    S = W @ W.T
    lam = np.linalg.eigh(S)[0][::-1]              # descending eigenvalues
    Uf, s, Vt = np.linalg.svd(W, full_matrices=True)
    bridge_gap = np.abs(lam[:d] - s**2).max()
    tail_max = np.abs(lam[d:]).max()
    lam_pad = np.concatenate([s**2, np.zeros(n - d)])
    pad_gap = np.abs(Uf @ np.diag(lam_pad) @ Uf.T - S).max()
    return bridge_gap, tail_max, pad_gap


bg, tm, pg = bridge_check(W)
print("bridge_gap:", bg, " tail_max:", tm, " pad_gap:", pg)
assert bg < 1e-8 and pg < 1e-8 and tm < 1e-10
print("contract satisfied")

### Checkpoint 7

1. The statement allowed `np.linalg.eig` nowhere.
   Would substituting `eig` for `eigh` have produced wrong *numbers*,
   or a different kind of trouble?
   Name the two output-hygiene chores `eig` would add.
2. Inside `bridge_check`, could the thin call
   (`full_matrices=False`) have computed `bridge_gap` and `tail_max`
   correctly?
   And `pad_gap`?

## 8. Common Pitfalls III

**Pitfall 1: treating `Vt` as $V$.**
The third output is *already* $V^{\mathsf T}$.
Using `Vt.T` where `Vt` belongs (or vice versa) is the most common SVD
bug in existence — and on **square** matrices the shapes still align,
so nothing crashes and the numbers are silently wrong.
(On rectangular matrices you often get lucky: the shape error crashes
and outs the bug.)

In [ ]:
Wsq = np.array([[2., 1., 0.], [0., 1., 3.], [1., 0., 1.]])   # square (3,3)
U, s, Vt = np.linalg.svd(Wsq, full_matrices=False)
right = np.abs(U @ np.diag(s) @ Vt - Wsq).max()
wrong = np.abs(U @ np.diag(s) @ Vt.T - Wsq).max()   # V where V^T belongs
print("correct  Vt   reconstruction gap:", right)
print("wrong    Vt.T reconstruction gap:", wrong, " <- no crash, just wrong")

Fix: recite the output contract — "U, sigma **vector**, V-**transpose**" —
and always run the reconstruction gap after unpacking an SVD.

**Pitfall 2: NumPy's default is `full_matrices=True`.**
Call `np.linalg.svd(W)` bare on a $(200, 40)$ matrix and `U` comes
back $(200, 200)$ — 160 columns you may not want, silently.
Symptom: memory jumps, or a later `U[:, :r] @ ...` works by luck while
`U @ np.diag(s)` crashes with a shape error.
Fix: pass the flag **explicitly, every call** — thin unless you need
whole-space work (this course's taught default is
`full_matrices=False`).

**Pitfall 3: expecting signed singular values.**
$\sigma_k \ge 0$ by definition; "the SVD found a negative stretch" is
always a misread.
The sign of a stretch lives in the *directions* ($u_k$, $v_k$ may
flip), and flipping **both** $u_k$ and $v_k$ changes nothing at all —
the SVD's version of the eigenvector sign freedom, governed by the
same Session 2 pin when comparing routes.

**Pitfall 4: `s` is a vector.**
`U @ s @ Vt` broadcasts into nonsense or crashes;
the sandwich needs `np.diag(s)`, the atom sum and `U * s` are the
vector-friendly forms (Session 2 §7's idiom).

**Pitfall 5: reordering `s`.**
`np.linalg.svd` already returns descending order — applying the `eigh`
reorder idiom here *reverses a correct order into ascending*.
The two calls have opposite conventions; the idiom belongs to
`eigh`/`eig` only.

### Checkpoint 8

1. Bare call: `U, s, Vt = np.linalg.svd(np.ones((50, 3)))`.
   What are the three shapes?
   Which flag was silently in force?
2. A teammate "fixes" surprising results by writing
   `s = np.sort(s)[::-1]` after every `svd` call and
   `vals = vals[::-1]` after every `eigh` call, but refuses to touch
   eigenvector or singular-vector arrays.
   For each of the two habits, say whether it is a no-op, a fix, or a
   new bug — and why.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $Wq$ is a 6-vector but $\lambda q$ is a 4-vector — the two sides of
   the equation live in different spaces, so equality cannot even be
   asked.
2. $WW^{\mathsf T}$ is $(6, 6)$; $W^{\mathsf T}W$ is $(4, 4)$.
   F3 calls $WW^{\mathsf T}$ the Gram matrix (of the rows) — all
   pairwise row dot products.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Rank 1 ⇒ exactly one nonzero singular value, so
   $\sigma = (\sigma_1, 0)$ with
   $\sigma_1^2 = 1 + 4 + 4 + 16 = 25$: $\sigma = (5, 0)$.
2. $\lVert Wv_k \rVert = \lVert \sigma_k u_k \rVert = \sigma_k$ (unit
   $u_k$, and $\sigma_k \ge 0$).
   Among the perpendicular directions $v_k$ — which capture the whole
   map — the largest stretch is $\sigma_1$; Session 2 §6's
   weighted-average argument (applied to $W^{\mathsf T}W$) upgrades
   this to *all* unit inputs.
3. $U\Sigma V^{\mathsf T}
   = I \begin{pmatrix} 3 & 0 \\ 0 & 2 \end{pmatrix}
   \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}
   = \begin{pmatrix} 3 & 0 \\ 0 & -2 \end{pmatrix} = W$. ✓

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Math's $V^{\mathsf T}V = I$ is code's `Vt @ (Vt).T` — `Vt` *is*
   $V^{\mathsf T}$, so `Vt @ Vt.T` computes
   $V^{\mathsf T}(V^{\mathsf T})^{\mathsf T} = V^{\mathsf T}V$.
   (`Vt.T @ Vt` is $VV^{\mathsf T}$ — the wrong-side product, not $I$
   in the wide/rank-deficient cases.)
2. It computed $U\Sigma V$ (an un-transposed $V$ on the right).
   The reconstruction gap jumps from $\sim 10^{-15}$ to order 1 —
   which is exactly why the gap check is run after every unpack.
3. None of them — `svd` already returns descending `s` with matching
   columns/rows.
   (The reorder idiom belongs to `eig`/`eigh`.)

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $(100, 7)$: thin — `U (100, 7)`, `s (7,)`, `Vt (7, 7)`;
   full — `U (100, 100)`, `s (7,)`, `Vt (7, 7)`.
2. $(7, 100)$: thin — `U (7, 7)`, `s (7,)`, `Vt (7, 100)`;
   full — `U (7, 7)`, `s (7,)`, `Vt (100, 100)`.
3. Shape table: `s` has $\min(6, 4) = 4$ entries, so at most 4 can be
   nonzero.
   Rank bound: $\operatorname{rank}(W) \le \min(n, d) = 4$, and the
   nonzero-$\sigma$ count *is* the rank.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. In collapsing $\sum_k \sigma_k v_k (u_k \cdot u_i)$ to
   $\sigma_i v_i$: every dot $u_k \cdot u_i$ with $k \ne i$ vanished.
   Without orthonormality the result would stay a mixture
   $\sum_k \sigma_k (u_k \cdot u_i)\, v_k$ of all the $v_k$ — no clean
   eigen-equation.
2. $S$ is $(200, 200)$: 200 eigenvalues — the 40 values
   $\sigma_1^2, \dots, \sigma_{40}^2 > 0$ plus $0$ with multiplicity
   $160$.
3. They would have to span the $(n - d)$-dimensional null space of $S$
   — the directions perpendicular to everything $W$ produces — but thin
   $U$'s $d$ columns all lie *inside* the span of $W$'s outputs;
   $UU^{\mathsf T} \ne I_n$ is the numerical witness.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. For a wide $(3, 5)$ matrix the roles mirror: $WW^{\mathsf T}$
   is $(3, 3)$ and fully covered by the 3 singular values, while
   $W^{\mathsf T}W$ is $(5, 5)$ and needs $5 - 3 = 2$ padded zeros
   (with full `Vt` supplying the whole-space basis).
2. Exact zeros surface as $\pm 10^{-16}$-scale float fuzz — Session 2
   §6's numerical footnote; treat $|\lambda| < \text{tol}$ as zero.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Same numbers eventually, but two extra chores: strip the complex
   dtype (check `.imag`, keep `.real`) and apply the argsort reorder
   (`eig` is unsorted, `eigh` is ascending-sorted so a plain `[::-1]`
   suffices) — plus `eig` does not guarantee orthonormal columns to
   machine precision the way `eigh` does.
   And under the statement's ban it scores zero regardless.
2. `bridge_gap` and `tail_max`: yes — they only need `s` and the
   `eigh` route.
   `pad_gap`: no — it sandwiches an $(n, n)$ diagonal between $U$ and
   $U^{\mathsf T}$, which requires the full $(n, n)$ $U$; thin $U$
   gives a shape error (or, if you pad sloppily, a wrong matrix).

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. `U (50, 50)`, `s (3,)`, `Vt (3, 3)` — `full_matrices=True`, NumPy's
   silent default.
2. The `svd` habit is a no-op (`s` is already descending — sorting
   changes nothing).
   The `eigh` habit is a **new bug**: it correctly reverses the values
   but abandons the columns of `vecs`, so every eigenpair is
   mismatched — exactly Session 2's Bug A, caught by the
   eigen-equation residual.

</details>